In [61]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [62]:
import pandas as pd
from CP2.customer_cleaner import CustomerCleaner

df = pd.read_csv("../../dataset/telco_customer.csv")

customclean_obj = CustomerCleaner(data_frame=df)
df = customclean_obj.clean()

In [63]:
print(f"Total Columns : {len(df.columns.tolist())}")

Total Columns : 21


In [64]:
# 27. Feature 1 - tenure_bucket

bins = [0, 12, 24, 48, 72]
labels = ["0-12", "13-24", "25-48", "49-72"]
df["tenure_bucket"] = pd.cut(
    df["tenure"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [65]:
# 28. Feature 2 - high_change_flag

median = df["monthly_charges"].mean()
def fill_high_charge(charge):
    return 1 if charge > median else 0

df["high_charge_flag"] = df["monthly_charges"].apply(fill_high_charge)

In [66]:
df["online_security"]

0        No
1       Yes
2       Yes
3       Yes
4        No
       ... 
7038    Yes
7039     No
7040    Yes
7041     No
7042    Yes
Name: online_security, Length: 7043, dtype: str

In [67]:
# 29. Feature 3 - serivce_count

def fill_service_count(row):
    service_columns = [
        "online_security",
        "online_backup",
        "device_protection",
        "tech_support",
        "streaming_tv",
        "streaming_movies"
    ]
    count = 0
    for cols in service_columns:
        count += 1 if row[cols] == "Yes" else 0
    return count

df["service_count"] = df.apply(fill_service_count, axis=1)

In [68]:
df["service_count"]

0       1
1       2
2       2
3       3
4       0
       ..
7038    5
7039    4
7040    1
7041    0
7042    5
Name: service_count, Length: 7043, dtype: int64

In [69]:
df.columns

Index(['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents',
       'tenure', 'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'total_charges', 'churn',
       'tenure_bucket', 'high_charge_flag', 'service_count'],
      dtype='str')

In [70]:
# 30. Feature 4 - is_long_term_customer

def fill_is_long_term_customer(tenure):
    return 1 if tenure >= 24 else 0

df["is_long_term_customer"] = df["tenure"].apply(fill_is_long_term_customer)

In [71]:
df["is_long_term_customer"].head()

0    0
1    1
2    0
3    1
4    0
Name: is_long_term_customer, dtype: int64

In [72]:
# 31. Feature 5 - has_streaming_bundle

def fill_has_streaming_bundle(row):
    return 1 if row["streaming_tv"] == "Yes" and row["streaming_movies"] == "Yes" else 0

df["has_streaming_bundle"] = df[["streaming_tv", "streaming_movies"]].apply(fill_has_streaming_bundle, axis = 1)

In [73]:
df["payment_method"].unique()

<StringArray>
[         'Electronic check',              'Mailed check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str

In [74]:
# 32. Feature 6 - auto_pay_flag

def fill_auto_pay_flag(value):
    return 1 if "automatic" in value else 0

df["auto_pay_flag"] = df["payment_method"].apply(fill_auto_pay_flag)

In [75]:
cols = ["high_charge_flag", "service_count", "is_long_term_customer", "has_streaming_bundle", "auto_pay_flag", "churn"]
cor = df[cols].corr()

In [76]:
cor["churn"]

high_charge_flag         0.203245
service_count           -0.087698
is_long_term_customer   -0.311496
has_streaming_bundle     0.040441
auto_pay_flag           -0.209902
churn                    1.000000
Name: churn, dtype: float64

In [77]:
print(f"Update Columns : {len(df.columns.tolist())}")

Update Columns : 27


In [83]:
assert df["monthly_charges"].notna().all(), \
            "monthly_charges contains null values"

In [80]:
assert df["churn"].isin([0, 1]).all(), \
            "churn contains values other than 0 or 1"

In [60]:
# 34. save customer_features.csv

df.to_csv("../../dataset/customer_features_from_cp4.csv")